# HW3 — Two retrievers and a model: what RAG actually sends

**Name:** Ermekbayeva Indira

**Student ID:** S23068086

**Group:** 8

**Repository:** https://github.com/indiraermekbayeva-lgtm/hw3-Indi.git

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

> I used ChatGPT to help explain the assignment, write and debug Python code, and organize the analysis. I ran the experiments in Google Colab and checked the results against the reference answers.

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

In [43]:
import os
import shutil

os.makedirs("/content/data", exist_ok=True)

for filename in ["corpus.jsonl", "queries.json", "qa.json"]:
    source = f"/content/{filename}"
    destination = f"/content/data/{filename}"

    if os.path.exists(source):
        shutil.copy2(source, destination)

print("Files ready")

Files ready


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** (9 + 10 + 6) / 3 = 25 / 3 = 8.3333

**Query terms:** how, does, rag, retrieve, information

**TF, DF and IDF of each query term:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| does | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | 1.9459 |
| information | 0 | 0 | 0 | 0 | 1.9459 |

**IDF calculation:**

N = 3

For every query term, n(t) = 0.

idf(t) = ln((N - n(t) + 0.5) / (n(t) + 0.5))

idf(t) = ln((3 - 0 + 0.5) / (0 + 0.5))

idf(t) = ln(3.5 / 0.5) = ln(7) = 1.9459

**BM25 of each document** (write out the sum for every document, one line per term that contributes):

k1 = 1.5, b = 0.75

BM25(q,d) = sum of:

idf(t) * f(t,d) * (k1 + 1) /
(f(t,d) + k1 * (1 - b + b * |d| / avgdl))

All query terms have f(t,d) = 0 in every document.

Therefore, every term contributes zero to the BM25 score.

D1 = 0 + 0 + 0 + 0 + 0 = 0

D2 = 0 + 0 + 0 + 0 + 0 = 0

D3 = 0 + 0 + 0 + 0 + 0 = 0

| Document | BM25 Score | Rank |
|---|---:|---|
| D1 | 0 | Tied 1st |
| D2 | 0 | Tied 1st |
| D3 | 0 | Tied 1st |

**Explanation:**

All three documents have a BM25 score of 0 because none of the query terms exactly matches the document tokens.

For example, the query contains "retrieve", but D2 contains "retrieval" and "retrieves". Without stemming, BM25 treats these as different words.

Therefore, BM25 cannot distinguish between the three documents, and all three have the same rank.

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, gener, text, and, answer, questions | 9 |
| D2 | retriev, augmented, gener, retriev, relevant, documents, before, gener, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** (9 + 10 + 6) / 3 = 25 / 3 = 8.3333

**Stemmed query terms:**

how, doe, rag, retriev, inform

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| doe | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retriev | 0 | 2 | 0 | 1 | 0.5108 |
| inform | 0 | 0 | 0 | 0 | 1.9459 |

**IDF calculations:**

N = 3

For terms with DF = 0:

idf(t) = ln((3 - 0 + 0.5) / (0 + 0.5))

= ln(7) = 1.9459

For "retriev", DF = 1:

idf(retriev) = ln((3 - 1 + 0.5) / (1 + 0.5))

= ln(2.5 / 1.5) = 0.5108

**BM25 of each document**, worked:

k1 = 1.5, b = 0.75

BM25(q,d) = sum of:

idf(t) * f(t,d) * (k1 + 1) /
(f(t,d) + k1 * (1 - b + b * |d| / avgdl))

**D1:**

All query terms have TF = 0.

BM25(D1) = 0 + 0 + 0 + 0 + 0 = 0

**D2:**

Only "retriev" contributes to the score.

f(retriev, D2) = 2

BM25(D2) = 0.5108 * (2 * 2.5) /
(2 + 1.5 * (0.25 + 0.75 * 10 / 8.3333))

= 0.5108 * 5 / 3.725

= 0.6857

**D3:**

All query terms have TF = 0.

BM25(D3) = 0 + 0 + 0 + 0 + 0 = 0

| Document | BM25 Score | Rank |
|---|---:|---|
| D1 | 0.0000 | 2 (tied) |
| D2 | 0.6857 | 1 |
| D3 | 0.0000 | 2 (tied) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

In A1(a), all three documents have a BM25 score of 0, so they are tied. This happens because none of the query terms exactly matches the document tokens.

In A1(b), D2 ranks first with a BM25 score of 0.6857. Stemming changes "retrieve", "retrieves", and "retrieval" into "retriev", allowing the query to match D2.

Therefore, stemming helps BM25 recognize different forms of the same word and improves retrieval.

## A2 — cosine similarity

**Vectors:**

Query = [0.8, 0.2, 0.5]  
D1 = [0.7, 0.1, 0.4]  
D2 = [0.9, 0.3, 0.6]  
D3 = [0.1, 0.8, 0.2]

**Formula:**

cos(Q,D) = (Q · D) / (||Q|| × ||D||)

**Query magnitude:**

||Q|| = sqrt(0.8² + 0.2² + 0.5²) = sqrt(0.93) = 0.9644

**Document magnitudes:**

||D1|| = sqrt(0.49 + 0.01 + 0.16) = sqrt(0.66) = 0.8124

||D2|| = sqrt(0.81 + 0.09 + 0.36) = sqrt(1.26) = 1.1225

||D3|| = sqrt(0.01 + 0.64 + 0.04) = sqrt(0.69) = 0.8307

**Cosine similarity calculations:**

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8×0.7 + 0.2×0.1 + 0.5×0.4 = 0.78 | 0.9644 | 0.8124 | 0.9956 |
| Query · D2 | 0.8×0.9 + 0.2×0.3 + 0.5×0.6 = 1.08 | 0.9644 | 1.1225 | 0.9977 |
| Query · D3 | 0.8×0.1 + 0.2×0.8 + 0.5×0.2 = 0.34 | 0.9644 | 0.8307 | 0.4244 |

cos(Q,D1) = 0.78 / sqrt(0.93 × 0.66) = 0.9956

cos(Q,D2) = 1.08 / sqrt(0.93 × 1.26) = 0.9977

cos(Q,D3) = 0.34 / sqrt(0.93 × 0.69) = 0.4244

**Ranking:**

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> D2 is the most similar document, with a cosine similarity of 0.9977.

**Is the ranking the same as BM25's? Compare it with both A1 rankings.**

> In A1(a), all documents were tied with BM25 scores of 0.
  In A1(b), D2 ranked first, while D1 and D3 were tied.
  Cosine similarity also ranks D2 first, but gives D1 second place and D3 third place.

**Why can BM25 and embedding retrieval disagree?**

> BM25 matches exact words. The query contains "retrieve", while D2 contains "retrieval" and "retrieves", so BM25 without stemming cannot match them.
Embedding retrieval compares vector similarity and can recognize related meanings without exact word matches.
Therefore, embeddings can identify D2 as relevant even when BM25 without stemming cannot.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [ ]:
import numpy as np

# A1(a) — BM25 without stemming
print("A1(a): BM25 without stemming")
print("D1 = 0")
print("D2 = 0")
print("D3 = 0")

# A1(b) — BM25 with stemming
N = 3
k1 = 1.5
b = 0.75
avgdl = 25 / 3

idf = np.log((N - 1 + 0.5) / (1 + 0.5))

def bm25(tf, doc_len):
    return idf * (tf * (k1 + 1)) / (
        tf + k1 * (1 - b + b * doc_len / avgdl)
    )

print("\nA1(b): BM25 with stemming")
print("D1 =", 0)
print("D2 =", round(bm25(2, 10), 4))
print("D3 =", 0)

# A2 — Cosine similarity
query = np.array([0.8, 0.2, 0.5])

docs = {
    "D1": np.array([0.7, 0.1, 0.4]),
    "D2": np.array([0.9, 0.3, 0.6]),
    "D3": np.array([0.1, 0.8, 0.2])
}

print("\nA2: Cosine similarity")

for name, doc in docs.items():
    dot = np.dot(query, doc)
    q_norm = np.linalg.norm(query)
    d_norm = np.linalg.norm(doc)
    cosine = dot / (q_norm * d_norm)

    print(f"{name}: {cosine:.4f}")

A1(a): BM25 without stemming
D1 = 0
D2 = 0
D3 = 0

A1(b): BM25 with stemming
D1 = 0
D2 = 0.6857
D3 = 0

A2: Cosine similarity
D1: 0.9956
D2: 0.9977
D3: 0.4244


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [2]:
import json
from pathlib import Path

with open("data/corpus.jsonl", "r", encoding="utf-8") as f:
    corpus = [json.loads(line) for line in f]

with open("data/queries.json", "r", encoding="utf-8") as f:
    queries = json.load(f)

print("Total chunks:", len(corpus))
print("Total queries:", len(queries))
print("First chunk:", corpus[0])

Total chunks: 74
Total queries: 2
First chunk: {'id': 'KZ-01', 'source': 'wikipedia', 'title': 'Kazakhstan (part 1)', 'lang': 'en', 'url': 'https://en.wikipedia.org/wiki/Kazakhstan', 'text': "Kazakhstan, officially the Republic of Kazakhstan, is a landlocked country situated primarily in Central Asia, with a portion of its territory extending into Eastern Europe. It borders Russia to the north and west, China to the east, Kyrgyzstan to the southeast, Uzbekistan to the south, and Turkmenistan to the southwest, and it has a coastline along the Caspian Sea. The capital is Astana, and the country's largest city and principal cultural and economic center is Almaty, which served as the capital until 1997."}


In [3]:
print(queries.keys())
print("Total queries:", len(queries["queries"]))

dict_keys(['note', 'queries'])
Total queries: 8


## B1 — BM25 with `rank_bm25`

**My tokenisation:** (lowercase? regex? stemming? stop words?)

> I use lowercase text and regex tokenization (\w+). I do not use stemming or remove stop words.

In [5]:
%pip install rank_bm25

In [21]:
import re
import numpy as np
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"\w+", text.lower())

tokenized_corpus = [tokenize(doc["text"]) for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

def bm25_search(query, top_k=3):
    scores = bm25.get_scores(tokenize(query))
    top_indices = np.argsort(scores)[::-1][:top_k]

    return [
        (corpus[i], float(scores[i]))
        for i in top_indices
    ]

for q in queries["queries"]:
    print(f"\n{q['id']}: {q['text']}")

    results = bm25_search(q["text"])

    for rank, (doc, score) in enumerate(results, 1):
        first_line = doc["text"].split("\n")[0]
        print(
            f"{rank}. {doc['id']} | "
            f"{first_line} | Score: {score:.4f}"
        )

    relevant = set(q["relevant"])
    hit = any(doc["id"] in relevant for doc, _ in results)

    print("BM25 Hit@3:", hit)


Q-01: What is form GR-305 used for?
1. GO-08 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case. | Score: 15.8255
2. GO-12 | A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days. An appeal reviews the decision on the evidence that was on file; new income information cannot be added through an appeal and needs a new application. | Score: 8.1953
3. GO-06 | Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a letter is not an application. GR-104 is

In [22]:
print(queries["queries"][0])

{'id': 'Q-01', 'text': 'What is form GR-305 used for?', 'relevant': ['GO-08', 'GO-12', 'GO-23']}


## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [8]:
%pip install "sentence-transformers>=5.0.0"

In [12]:
from huggingface_hub import login
login()

In [23]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer("google/embeddinggemma-300m")

texts = [doc["text"] for doc in corpus]

document_embeddings = model.encode_document(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedded chunks:", len(document_embeddings))

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embedded chunks: 74


In [24]:
query_embeddings = model.encode_query(
    [q["text"] for q in queries["queries"]],
    convert_to_numpy=True,
    show_progress_bar=True
)

similarities = cosine_similarity(query_embeddings, document_embeddings)

for i, q in enumerate(queries["queries"]):
    top_indices = np.argsort(similarities[i])[::-1][:3]

    print(f"\n{q['id']}: {q['text']}")
    for rank, idx in enumerate(top_indices, 1):
        doc = corpus[idx]
        print(
            f"{rank}. {doc['id']} | "
            f"{doc['text'].splitlines()[0][:120]} | "
            f"score={similarities[i][idx]:.4f}"
        )

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q-01: What is form GR-305 used for?
1. GO-08 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used  | score=0.6955
2. GO-12 | A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using for | score=0.4609
3. GO-23 | Если студент не согласен с решением, он может подать апелляцию по форме GR-305 в течение 14 календарных дней со дня реше | score=0.4505

Q-02: My family is poor. Is there financial help for students like me?
1. GO-01 | The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a stu | score=0.4446
2. GO-10 | Proof of income must accompany form GR-210. Accepted: a salary certificate from the employer issued within the last thre | score=0.4117
3. GO-20 | The need-based grant is for full-time undergraduate students only. Part-time students, distance-learning students and in | score=0.4090

Q-03: Өт

In [25]:
for i, q in enumerate(queries["queries"]):
    top_indices = np.argsort(similarities[i])[::-1][:3]
    top_ids = [corpus[idx]["id"] for idx in top_indices]
    relevant = q["relevant"]

    hit = any(doc_id in relevant for doc_id in top_ids)

    print(f"{q['id']}: hit@3 = {hit}")

Q-01: hit@3 = True
Q-02: hit@3 = True
Q-03: hit@3 = True
Q-04: hit@3 = True
Q-05: hit@3 = True
Q-06: hit@3 = True
Q-07: hit@3 = True
Q-08: hit@3 = True
Q-09: hit@3 = True
Q-10: hit@3 = True


## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | Which documents are required for form GR-104? | GO-06 | BM25, because the exact form number GR-104 appears in the document. |
| Q-10 | What should I do if I disagree with the Grant Office's decision? | GO-08 | EmbeddingGemma, because it can match the meaning of disagreement and appeal. |


In [26]:
# Add the two own queries to the original eight.
# Safe to rerun: the list will not accumulate duplicates.

queries["queries"] = queries["queries"][:8] + [
    {
        "id": "Q-09",
        "text": "Which documents are required for form GR-104?",
        "relevant": ["GO-06"]
    },
    {
        "id": "Q-10",
        "text": "What should I do if I disagree with the Grant Office's decision?",
        "relevant": ["GO-08"]
    }
]

print("Total queries:", len(queries["queries"]))
for q in queries["queries"]:
    print(q["id"], q["text"], "| relevant:", q["relevant"])


Total queries: 10
Q-01 What is form GR-305 used for? | relevant: ['GO-08', 'GO-12', 'GO-23']
Q-02 My family is poor. Is there financial help for students like me? | relevant: ['GO-01', 'GO-02', 'GO-03']
Q-03 Өтінішті қашанға дейін тапсыру керек? | relevant: ['GO-09', 'GO-14']
Q-04 What is the minimum GPA for the grant? | relevant: ['GO-02']
Q-05 When was Astana renamed Nur-Sultan, and why? | relevant: ['KZ-05']
Q-06 Where did the Soviet Union test its nuclear weapons? | relevant: ['KZ-37', 'KZ-38']
Q-07 Can the grant be sent to my mother's card? | relevant: ['GO-15']
Q-08 If my grades drop in the autumn, do I still get paid in February? | relevant: ['GO-17']
Q-09 Which documents are required for form GR-104? | relevant: ['GO-06']
Q-10 What should I do if I disagree with the Grant Office's decision? | relevant: ['GO-08']


## Results for all ten or more queries: top 3 from each retriever

In [44]:
query_embeddings = model.encode_query(
    [q["text"] for q in queries["queries"]],
    convert_to_numpy=True,
    show_progress_bar=False
)

similarities = cosine_similarity(query_embeddings, document_embeddings)
for i, q in enumerate(queries["queries"]):
    bm25_results = bm25_search(q["text"], top_k=3)
    bm25_ids = [doc["id"] for doc, score in bm25_results]
    bm25_hit = any(doc_id in q["relevant"] for doc_id in bm25_ids)

    embedding_indices = np.argsort(similarities[i])[::-1][:3]
    embedding_ids = [corpus[idx]["id"] for idx in embedding_indices]
    embedding_hit = any(
        doc_id in q["relevant"] for doc_id in embedding_ids
    )

    print(f"\n{q['id']}: {q['text']}")
    print(f"BM25 top 3: {bm25_ids}")
    print(f"BM25 hit@3: {bm25_hit}")
    print(f"Embedding top 3: {embedding_ids}")
    print(f"Embedding hit@3: {embedding_hit}")


Q-01: What is form GR-305 used for?
BM25 top 3: ['GO-08', 'GO-12', 'GO-06']
BM25 hit@3: True
Embedding top 3: ['GO-08', 'GO-12', 'GO-23']
Embedding hit@3: True

Q-02: My family is poor. Is there financial help for students like me?
BM25 top 3: ['KZ-17', 'GO-20', 'KZ-25']
BM25 hit@3: False
Embedding top 3: ['GO-01', 'GO-10', 'GO-20']
Embedding hit@3: True

Q-03: Өтінішті қашанға дейін тапсыру керек?
BM25 top 3: ['GO-14', 'GO-23', 'GO-22']
BM25 hit@3: True
Embedding top 3: ['GO-14', 'GO-09', 'GO-11']
Embedding hit@3: True

Q-04: What is the minimum GPA for the grant?
BM25 top 3: ['GO-05', 'GO-17', 'GO-19']
BM25 hit@3: False
Embedding top 3: ['GO-02', 'GO-05', 'GO-17']
Embedding hit@3: True

Q-05: When was Astana renamed Nur-Sultan, and why?
BM25 top 3: ['KZ-05', 'KZ-06', 'KZ-11']
BM25 hit@3: True
Embedding top 3: ['KZ-05', 'KZ-06', 'KZ-03']
Embedding hit@3: True

Q-06: Where did the Soviet Union test its nuclear weapons?
BM25 top 3: ['KZ-37', 'KZ-38', 'KZ-03']
BM25 hit@3: True
Embedding

## B3 — comparison

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | True | GO-08, GO-12, GO-23 | True |
| Q-02 | KZ-17, GO-20, KZ-25 | False | GO-01, GO-10, GO-20 | True |
| Q-03 | GO-14, GO-23, GO-22 | True | GO-14, GO-09, GO-11 | True |
| Q-04 | GO-05, GO-17, GO-19 | False | GO-02, GO-05, GO-17 | True |
| Q-05 | KZ-05, KZ-06, KZ-11 | True | KZ-05, KZ-06, KZ-03 | True |
| Q-06 | KZ-37, KZ-38, KZ-03 | True | KZ-37, KZ-38, KZ-10 | True |
| Q-07 | GO-21, GO-16, GO-01 | False | GO-15, GO-18, GO-01 | True |
| Q-08 | GO-17, GO-05, GO-04 | True | GO-17, GO-18, GO-15 | True |
| Q-09 | GO-06, GO-03, GO-19 | True | GO-06, GO-10, GO-14 | True |
| Q-10 | GO-11, GO-13, GO-02 | False | GO-08, GO-21, GO-12 | True |
| **hit@3 total** | | **6/10** | | **10/10** |

**Both retrievers agree:** which query, and why was it easy?

> Q-01. Both systems retrieve relevant documents GO-08 and GO-12. The query contains the exact form number "GR-305", which appears in the relevant documents. This makes the query easy for both retrievers.


**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

> In my current results, BM25 did not outperform EmbeddingGemma on hit@3 for any query. Q-09 demonstrates BM25's strength in exact keyword matching: it retrieves GO-06, which contains the exact identifier "GR-104" and describes the required documents. However, EmbeddingGemma also ranks GO-06 first, so BM25 did not win this query.

**Embeddings did better:** which query, and what could BM25 not match?

> Q-02. BM25 retrieves irrelevant documents about Lake Balkhash and beshbarmak, giving hit@3 = False. EmbeddingGemma retrieves GO-01, GO-10, and GO-20, giving hit@3 = True. BM25 fails to connect the informal phrase "My family is poor" with the official concept "need-based grant", while EmbeddingGemma matches their semantic meaning.

**Additional observation — Q-04:**
> BM25 ranks GO-05 first, but its title begins with "ARCHIVED" and describes the old 2025 rules. EmbeddingGemma ranks GO-02 first, which contains the current 2026 eligibility requirements. This shows why the retrieved document's content and title should be checked, not just its keyword score.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

## C1 — the complete prompt, printed before it is sent

In [31]:
import json

with open("/content/data/qa.json", "r", encoding="utf-8") as f:
    qa_data = json.load(f)

print("Number of questions:", len(qa_data["questions"]) if isinstance(qa_data, dict) else len(qa_data))
print(qa_data)

Number of questions: 6
{'note': "Part C's six questions, each asked of all three systems. `reference` is what the corpus says; `relevant` lists the chunks that hold it. C-06 has no answer in the corpus on purpose: the right answer from a RAG system is that it does not know.", 'questions': [{'id': 'C-01', 'question': 'How much does a student in income band 2 receive, and when is it paid?', 'reference': '150,000 tenge for the academic year, in two equal instalments: October and February.', 'relevant': ['GO-04', 'GO-15']}, {'id': 'C-02', 'question': 'My grant application was refused. How long do I have to appeal, and which form do I use?', 'reference': '14 calendar days from the date on the decision letter, using form GR-305.', 'relevant': ['GO-08', 'GO-12', 'GO-23']}, {'id': 'C-03', 'question': 'What is the minimum GPA needed to qualify for the need-based grant?', 'reference': 'A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)', 'relevant':

In [33]:
import tiktoken
import numpy as np

# C-01: question from qa.json
q = qa_data["questions"][0]
question = q["question"]

# Retrieve top 3 chunks using EmbeddingGemma
query_embedding = model.encode_query(
    [question],
    convert_to_numpy=True,
    show_progress_bar=False
)

scores = cosine_similarity(
    query_embedding,
    document_embeddings
)[0]

top_indices = np.argsort(scores)[::-1][:3]
retrieved_chunks = [corpus[i] for i in top_indices]

# System instructions
system_message = (
    "Answer the question using only the provided context.\n"
    "If the answer cannot be found in the context, say that you do not know.\n"
    "Cite the context blocks you used, like [1] or [2][3]."
)

# Numbered context blocks
context = "\n\n".join(
    f"[{i}] {chunk['id']}\n{chunk['text']}"
    for i, chunk in enumerate(retrieved_chunks, start=1)
)

# User message
user_message = (
    f"Context:\n{context}\n\n"
    f"Question: {question}"
)

# Print the complete prompt BEFORE sending it
print("========== SYSTEM MESSAGE ==========")
print(system_message)

print("\n========== USER MESSAGE ==========")
print(user_message)

# Count tokens using o200k_base
enc = tiktoken.get_encoding("o200k_base")

system_tokens = len(enc.encode(system_message))
question_tokens = len(enc.encode(question))
context_tokens = len(enc.encode(context))
total_tokens = (
    system_tokens
    + len(enc.encode(user_message))
)

print("\n========== TOKEN COUNTS ==========")
print("System instructions:", system_tokens)
print("Question:", question_tokens)
print("Retrieved context:", context_tokens)
print("Total input tokens (estimated):", total_tokens)

print("\n========== CHUNK SIZES ==========")
print("Number of retrieved chunks:", len(retrieved_chunks))

for i, chunk in enumerate(retrieved_chunks, start=1):
    text = chunk["text"]
    print(
        f"Chunk {i} ({chunk['id']}): "
        f"{len(enc.encode(text))} tokens, "
        f"{len(text)} characters"
    )

print(
    "Total context size:",
    context_tokens,
    "tokens,",
    len(context),
    "characters"
)

========== SYSTEM MESSAGE ==========
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

========== USER MESSAGE ==========
Context:
[1] GO-04
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] GO-03
The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. Households in band 3 are not eligible for the need-based grant, although th

In [35]:
%pip -q install openai

from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

print("OpenRouter client initialized")

OpenRouter client initialized


In [36]:
models = client.models.list()

matches = [
    m.id for m in models.data
    if "gpt-5.6" in m.id.lower() or "luna" in m.id.lower()
]

print("Matching models:", matches)

Matching models: ['openai/gpt-6-luna-pro', 'openai/gpt-6-luna-pro:batch', 'openai/gpt-6-luna', 'openai/gpt-6-luna:batch', '~openai/gpt-luna-latest', 'openai/gpt-5.6-luna-pro', 'openai/gpt-5.6-luna-pro:batch', 'openai/gpt-5.6-luna', 'openai/gpt-5.6-luna:batch', 'openai/gpt-5.6-terra-pro', 'openai/gpt-5.6-terra-pro:batch', 'openai/gpt-5.6-terra', 'openai/gpt-5.6-terra:batch', 'openai/gpt-5.6-sol-pro', 'openai/gpt-5.6-sol-pro:batch', 'openai/gpt-5.6-sol', 'openai/gpt-5.6-sol:batch', 'sao10k/l3-lunaris-8b']


In [37]:
response = client.chat.completions.create(
    model="openai/gpt-5.6-luna",
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ]
)

print("Answer:")
print(response.choices[0].message.content)

print("\nActual prompt tokens:")
print(response.usage.prompt_tokens)

Answer:
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]

Actual prompt tokens:
376


| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 44 |
| Question | 18 |
| Retrieved context | 300 |
| **Total input** | **366** |
| **`usage.prompt_tokens` from the response** | **376** |

- Number of retrieved chunks: 3
- Tokens per chunk: GO-04: 90 tokens; GO-03: 104 tokens; GO-15: 85 tokens.
- Total context size (tokens / characters): 300 tokens / 1,163 characters.

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

> The estimated input contains 366 tokens, while the API reports 376 tokens. The difference is 10 tokens because the API also counts message-formatting overhead, including roles and special tokens. The estimated total includes additional labels and separators.

**What exactly does retrieval add to the model's input?**

> Retrieval adds three numbered context blocks from the corpus: [1] GO-04, [2] GO-03, and [3] GO-15. Each block contains a document ID and its text. These blocks are inserted into the user message before the question. They provide information about grant amounts, income bands, and payment dates, allowing the model to answer using retrieved evidence and cite the relevant block.

## C2 — three systems, six questions

In [38]:
# C2 — Compare LLM only, BM25 RAG, and EmbeddingGemma RAG

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

MODEL_NAME = "openai/gpt-5.6-luna"

rag_system = (
    "Answer the question using only the provided context.\n"
    "If the answer cannot be found in the context, say that you do not know.\n"
    "Cite the context blocks you used, like [1] or [2][3]."
)

def ask_llm(question, chunks=None):
    if chunks is None:
        messages = [
            {"role": "user", "content": question}
        ]
    else:
        context = "\n\n".join(
            f"[{i}] {doc['id']}\n{doc['text']}"
            for i, doc in enumerate(chunks, start=1)
        )
        messages = [
            {"role": "system", "content": rag_system},
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}"
            }
        ]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages
    )
    return response.choices[0].message.content


def embedding_top3(question):
    query_vector = model.encode_query(
        [question],
        convert_to_numpy=True,
        show_progress_bar=False
    )
    scores = cosine_similarity(query_vector, document_embeddings)[0]
    indices = np.argsort(scores)[::-1][:3]
    return [corpus[i] for i in indices]


c2_results = {}

for item in qa_data["questions"]:
    qid = item["id"]
    question = item["question"]

    bm25_chunks = [
        doc for doc, score in bm25_search(question, top_k=3)
    ]
    embedding_chunks = embedding_top3(question)

    llm_answer = ask_llm(question)
    bm25_answer = ask_llm(question, bm25_chunks)
    embedding_answer = ask_llm(question, embedding_chunks)

    c2_results[qid] = {
        "reference": item["reference"],
        "llm_only": llm_answer,
        "bm25": bm25_answer,
        "embedding": embedding_answer,
        "bm25_ids": [doc["id"] for doc in bm25_chunks],
        "embedding_ids": [doc["id"] for doc in embedding_chunks]
    }

    print(f"\n{'='*60}")
    print(f"{qid}: {question}")
    print(f"REFERENCE: {item['reference']}")
    print(f"\nLLM ONLY: {llm_answer}")
    print(f"\nBM25 RAG: {bm25_answer}")
    print(f"\nEMBEDDING RAG: {embedding_answer}")
    print(f"{'='*60}")


C-01: How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.

LLM ONLY: Which student support scheme or institution do you mean? “Income band 2” is used by several bursary and financial-aid schemes, and the amount and payment date vary. Please share the scheme, school/university, or a link.

BM25 RAG: A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [1]

EMBEDDING RAG: A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]

C-02: My grant application was refused. How long do I have to appeal, and which form do I use?
REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305.

LLM ONLY: The deadline and appeal for

Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ Asked for clarification | ✓ 150,000 tenge, Oct/Feb | ✓ 150,000 tenge, Oct/Feb |
| C-02 | ✗ No exact deadline or form | ✓ 14 days, GR-305 | ✓ 14 days, GR-305 |
| C-03 | ✗ No exact GPA | ✓ GPA 2.67 | ✓ GPA 2.67 |
| C-04 | ✓ October 2017, 2031 | ✓ October 2017, 2031 | ✓ October 2017, 2031 |
| C-05 | ✓ 1963, Alma-Ata Institute of National Economy | ✓ 1963, Almaty Institute of National Economy | ✓ 1963, Almaty Institute of National Economy |
| C-06 | ✗ General advice, no corpus-based answer | ✓ Amount unknown, separate scheme | ✓ Amount unknown, separate scheme |
| **Agrees with reference** | **2/6** | **6/6** | **6/6** |

## C3 — question 1 of 3: C-03

**Question:** What is the minimum GPA needed to qualify for the need-based grant?

**1. What BM25 retrieved** (top 3, ids and scores):

| Rank | Chunk ID | Score |
|---|---|---:|
| 1 | GO-16 | 15.7985 |
| 2 | GO-02 | 14.0587 |
| 3 | GO-06 | 13.8516 |

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

| Rank | Chunk ID | Score |
|---|---|---:|
| 1 | GO-02 | 0.6286 |
| 2 | GO-20 | 0.5691 |
| 3 | GO-05 | 0.5451 |

**3. What was sent to the model** (the context blocks):

> The complete context blocks for both RAG systems are printed in the code output below.

- **BM25 RAG:** [1] GO-16, [2] GO-02, [3] GO-06.
- **EmbeddingGemma RAG:** [1] GO-02, [2] GO-20, [3] GO-05.

Both systems retrieved GO-02, which contains the correct GPA requirement of 2.67. EmbeddingGemma also retrieved GO-05, which contains the archived GPA requirement of 3.0.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> The LLM-only system did not provide the exact GPA and asked for clarification. BM25 RAG correctly answered 2.67 using GO-02 [2]. EmbeddingGemma RAG also correctly answered 2.67 using GO-02 [1]. Retrieval made both answers more accurate and grounded in the current 2026 rules. EmbeddingGemma ranked the correct document first, while BM25 ranked it second.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Neither RAG system missed the correct GPA. Both retrieved GO-02, which states: "their cumulative GPA is at least 2.67 on the 4.0 scale".
>
> However, EmbeddingGemma also retrieved GO-05, which contains the outdated rule: "Under the 2025 scheme the minimum GPA for the grant was 3.0".
>
> This was a retrieval-quality issue because an outdated document appeared in the top 3. It did not become a generation failure because the model correctly selected the current rule from GO-02. The LLM-only system could not identify the exact requirement without the relevant policy context.

In [39]:
# C3 — Question 1: C-03

q = next(
    item for item in qa_data["questions"]
    if item["id"] == "C-03"
)

question = q["question"]

print("QUESTION:", question)
print("REFERENCE:", q["reference"])

# BM25 top 3
bm25_results = bm25_search(question, top_k=3)

print("\n========== BM25 TOP 3 ==========")

for i, (doc, score) in enumerate(bm25_results, 1):
    print(f"{i}. {doc['id']} | Score: {score:.4f}")

# EmbeddingGemma top 3
query_vector = model.encode_query(
    [question],
    convert_to_numpy=True,
    show_progress_bar=False
)

scores = cosine_similarity(
    query_vector,
    document_embeddings
)[0]

top_indices = np.argsort(scores)[::-1][:3]
embedding_results = [(corpus[i], scores[i]) for i in top_indices]

print("\n========== EMBEDDING TOP 3 ==========")

for i, (doc, score) in enumerate(embedding_results, 1):
    print(f"{i}. {doc['id']} | Score: {score:.4f}")

# Full context blocks
for name, docs in [
    ("BM25", [doc for doc, score in bm25_results]),
    ("EmbeddingGemma", [doc for doc, score in embedding_results])
]:
    print(f"\n========== {name} CONTEXT ==========")

    for i, doc in enumerate(docs, 1):
        print(f"\n[{i}] {doc['id']}")
        print(doc["text"])

# Answers already generated in C2
print("\n========== MODEL ANSWERS ==========")

print("LLM ONLY:", c2_results["C-03"]["llm_only"])
print("BM25 RAG:", c2_results["C-03"]["bm25"])
print("EMBEDDING RAG:", c2_results["C-03"]["embedding"])

QUESTION: What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

========== BM25 TOP 3 ==========
1. GO-16 | Score: 15.7985
2. GO-02 | Score: 14.0587
3. GO-06 | Score: 13.8516

========== EMBEDDING TOP 3 ==========
1. GO-02 | Score: 0.6286
2. GO-20 | Score: 0.5691
3. GO-05 | Score: 0.5451

========== BM25 CONTEXT ==========

[1] GO-16
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] GO-02
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income b

## C3 — question 2 of 3: C-01

**Question:** How much does a student in income band 2 receive, and when is it paid?

**1. What BM25 retrieved** (top 3, ids and scores):

- GO-04 — 20.2192
- GO-02 — 18.7422
- GO-01 — 17.2430

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

- GO-04 — 0.5845
- GO-03 — 0.4842
- GO-15 — 0.4506

**3. What was sent to the model** (the context blocks):

> BM25: [1] GO-04, [2] GO-02, [3] GO-01.
>
> EmbeddingGemma: [1] GO-04, [2] GO-03, [3] GO-15.
>
> The full context texts are printed in the Code cell output.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> LLM-only asked for clarification. Both RAG systems correctly answered 150,000 tenge, paid in October and February. Retrieval made the answers more accurate and grounded in the documents.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Neither RAG system failed. Both retrieved GO-04, which states: "A student in band 2 receives 150,000 tenge for the academic year." Both models used this information correctly. There was no retrieval or generation failure.

In [40]:
# C3 — Question 2: C-01

q = next(item for item in qa_data["questions"] if item["id"] == "C-01")
question = q["question"]

print("QUESTION:", question)
print("REFERENCE:", q["reference"])

# BM25 top 3
bm25_results = bm25_search(question, top_k=3)

print("\n========== BM25 TOP 3 ==========")
for i, (doc, score) in enumerate(bm25_results, 1):
    print(f"{i}. {doc['id']} | Score: {score:.4f}")

# EmbeddingGemma top 3
query_vector = model.encode_query(
    [question],
    convert_to_numpy=True,
    show_progress_bar=False
)

scores = cosine_similarity(query_vector, document_embeddings)[0]
top_indices = np.argsort(scores)[::-1][:3]
embedding_results = [(corpus[i], scores[i]) for i in top_indices]

print("\n========== EMBEDDING TOP 3 ==========")
for i, (doc, score) in enumerate(embedding_results, 1):
    print(f"{i}. {doc['id']} | Score: {score:.4f}")

# Full context blocks
for name, results in [
    ("BM25", bm25_results),
    ("EmbeddingGemma", embedding_results)
]:
    print(f"\n========== {name} CONTEXT ==========")
    for i, (doc, score) in enumerate(results, 1):
        print(f"\n[{i}] {doc['id']}")
        print(doc["text"])

# Answers from C2
print("\n========== MODEL ANSWERS ==========")
print("LLM ONLY:", c2_results["C-01"]["llm_only"])
print("BM25 RAG:", c2_results["C-01"]["bm25"])
print("EMBEDDING RAG:", c2_results["C-01"]["embedding"])

QUESTION: How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.

========== BM25 TOP 3 ==========
1. GO-04 | Score: 20.2192
2. GO-02 | Score: 18.7422
3. GO-01 | Score: 17.2430

========== EMBEDDING TOP 3 ==========
1. GO-04 | Score: 0.5845
2. GO-03 | Score: 0.4842
3. GO-15 | Score: 0.4506

========== BM25 CONTEXT ==========

[1] GO-04
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] GO-02
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 

## C3 — question 3 of 3: C-06

**1. What BM25 retrieved** (top 3, ids and scores):

- GO-16 — 16.9958
- GO-01 — 14.1725
- GO-20 — 13.7202

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

- GO-20 — 0.6664
- GO-16 — 0.6161
- GO-01 — 0.5539

**3. What was sent to the model** (the context blocks):

> BM25: [1] GO-16, [2] GO-01, [3] GO-20.
>
> EmbeddingGemma: [1] GO-20, [2] GO-16, [3] GO-01.
>
> The complete context blocks are printed in the Code cell output.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> LLM-only gave general information about US financial aid instead of answering from the handbook.
>
> Both RAG systems correctly stated that the grant amount for master's students is unknown. BM25 cited [3], while EmbeddingGemma cited [1].
>
> EmbeddingGemma ranked the most relevant document, GO-20, first. BM25 ranked it third. Retrieval made both answers more grounded and prevented an unsupported grant amount.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> GO-20 states: "Master's and doctoral students are supported by a separate scheme, which this handbook does not describe".
>
> Both retrievers found GO-20, and both RAG systems correctly answered that the amount is unknown. Therefore, neither retrieval nor generation failed.
>
> The exact amount is missing from the corpus itself. This is a limitation of the available information, not a retrieval failure.
>
> The LLM-only system did not give the required corpus-based answer and introduced unrelated information about US financial aid.

In [41]:
# C3 — Question 3: C-06

q = next(item for item in qa_data["questions"] if item["id"] == "C-06")
question = q["question"]

print("QUESTION:", question)
print("REFERENCE:", q["reference"])

# BM25 top 3
bm25_results = bm25_search(question, top_k=3)

print("\n========== BM25 TOP 3 ==========")
for i, (doc, score) in enumerate(bm25_results, 1):
    print(f"{i}. {doc['id']} | Score: {score:.4f}")

# EmbeddingGemma top 3
query_vector = model.encode_query(
    [question],
    convert_to_numpy=True,
    show_progress_bar=False
)

scores = cosine_similarity(query_vector, document_embeddings)[0]
top_indices = np.argsort(scores)[::-1][:3]
embedding_results = [(corpus[i], scores[i]) for i in top_indices]

print("\n========== EMBEDDING TOP 3 ==========")
for i, (doc, score) in enumerate(embedding_results, 1):
    print(f"{i}. {doc['id']} | Score: {score:.4f}")

# Full context blocks
for name, results in [
    ("BM25", bm25_results),
    ("EmbeddingGemma", embedding_results)
]:
    print(f"\n========== {name} CONTEXT ==========")
    for i, (doc, score) in enumerate(results, 1):
        print(f"\n[{i}] {doc['id']}")
        print(doc["text"])

# Answers from C2
print("\n========== MODEL ANSWERS ==========")
print("LLM ONLY:", c2_results["C-06"]["llm_only"])
print("BM25 RAG:", c2_results["C-06"]["bm25"])
print("EMBEDDING RAG:", c2_results["C-06"]["embedding"])

QUESTION: I am a master's student. How much need-based grant can I get?
REFERENCE: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.

========== BM25 TOP 3 ==========
1. GO-16 | Score: 16.9958
2. GO-01 | Score: 14.1725
3. GO-20 | Score: 13.7202

========== EMBEDDING TOP 3 ==========
1. GO-20 | Score: 0.6664
2. GO-16 | Score: 0.6161
3. GO-01 | Score: 0.5539

========== BM25 CONTEXT ==========

[1] GO-16
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] GO-01
The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income household can pay for study costs w

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 finds documents based on matching words and their frequency. Embedding retrieval uses numerical vectors to compare the meaning of the question and documents, even when they use different words.

**2. When can BM25 perform better than embeddings?**

> BM25 can perform better when a question contains exact keywords, document IDs, or form codes. For example, in my Q-09 query about form GR-104, BM25 successfully retrieved the relevant document GO-06.

**3. When can embeddings perform better than BM25?**

> Embeddings perform better when the question and relevant document express the same meaning using different words. In my experiment, EmbeddingGemma achieved hit@3 on all ten queries, while BM25 missed four.

**4. Why is cosine similarity used with embeddings?**

> Cosine similarity measures how closely two vectors point in the same direction. It helps compare the semantic similarity between a question embedding and document embeddings.

**5. What information is actually sent to an LLM in a RAG system?**

> The LLM receives system instructions, the question, and the retrieved text chunks. In my notebook, each chunk includes a citation number, document ID, and text.

**6. Does the LLM search the original documents itself?**

> No. The retriever searches the document corpus and selects relevant chunks. The LLM receives these chunks in its prompt and generates an answer based on them.

**7. What happens if retrieval returns an irrelevant chunk?**

> An irrelevant chunk adds unnecessary information to the prompt and may confuse the model. This can lead to an incorrect or less accurate answer, especially if the relevant information is missing.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

> Increasing Top-K can improve RAG by providing more relevant information, but it can also add irrelevant or outdated documents that confuse the model.
>
> I tested question C-03 about the minimum GPA using EmbeddingGemma:
>
> - K = 1: GO-02. Answer: 2.67 GPA (correct).
> - K = 3: GO-02, GO-20, GO-05. Answer: 2.67 GPA (correct).
> - K = 8: GO-02, GO-20, GO-05, GO-16, GO-01, GO-06, GO-19, GO-17. Answer: 2.67 GPA (correct).
>
> All three answers were correct. Increasing K did not improve accuracy in this experiment because the correct information was already in the first chunk. However, K = 3 and K = 8 included the outdated GO-05 document, which could potentially confuse the model.

In [42]:
# Final Question 8 — Compare Top-K = 1, 3, 8
# Using C-03: minimum GPA

question = next(
    q["question"]
    for q in qa_data["questions"]
    if q["id"] == "C-03"
)

for k in [1, 3, 8]:
    chunks = embedding_top3(question) if k == 3 else None

    if chunks is None:
        query_vector = model.encode_query(
            [question],
            convert_to_numpy=True,
            show_progress_bar=False
        )
        scores = cosine_similarity(query_vector, document_embeddings)[0]
        indices = np.argsort(scores)[::-1][:k]
        chunks = [corpus[i] for i in indices]

    answer = ask_llm(question, chunks)

    print(f"\n===== TOP-K = {k} =====")
    print("Retrieved IDs:", [doc["id"] for doc in chunks])
    print("Answer:", answer)


===== TOP-K = 1 =====
Retrieved IDs: ['GO-02']
Answer: The minimum GPA needed is **2.67 on a 4.0 scale**. [1]

===== TOP-K = 3 =====
Retrieved IDs: ['GO-02', 'GO-20', 'GO-05']
Answer: The minimum GPA is **2.67 on a 4.0 scale** for the 2026 grant. [1]

===== TOP-K = 8 =====
Retrieved IDs: ['GO-02', 'GO-20', 'GO-05', 'GO-16', 'GO-01', 'GO-06', 'GO-19', 'GO-17']
Answer: The minimum GPA needed to qualify for the 2026 need-based grant is **2.67 on a 4.0 scale**. [1]


**9. Is a vector database required to build a RAG system?**

> No, a vector database is not required. In my notebook, I stored document embeddings in a NumPy array and used cosine similarity to retrieve relevant chunks. A vector database becomes more useful for large datasets and efficient searching.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

> EmbeddingGemma performed better in retrieval. It achieved hit@3 of 10/10 (100%), while BM25 achieved 6/10 (60%) across ten queries.
>
> In C2, both BM25 RAG and EmbeddingGemma RAG agreed with the reference answers on all six questions (6/6). Therefore, EmbeddingGemma was better at retrieving relevant chunks, but both RAG systems achieved the same answer accuracy in C2.

---
# Conclusion

> In this assignment, I compared BM25 and EmbeddingGemma retrieval and connected both to an LLM using RAG. EmbeddingGemma achieved higher retrieval accuracy, while both RAG systems answered all six C2 questions correctly.
>
> Next, I would build a hybrid RAG system combining BM25 and embeddings. This could improve retrieval by using both exact keyword matching and semantic similarity. I would also add a filter for outdated documents, because archived rules such as GO-05 could lead to incorrect answers.